# KEY-DOOR GRID WORLD: HUMAN PLAY
Dùng các nút **Lên / Xuống / Trái / Phải** để di chuyển trên bản đồ. Mục tiêu là lấy chìa khóa, mở cửa, lấy chìa khóa tái tạo và đi đến đích.

## Cách chơi
1. Chọn bản đồ.
2. Nhấn **Bắt đầu / Chơi lại**.
3. Dùng 4 nút điều khiển để di chuyển.
4. Theo dõi trạng thái chìa khóa, cửa, điểm thưởng và sự kiện bên dưới bản đồ.


**Điều khiển bàn phím:** W/A/S/D hoặc các phím mũi tên. Nếu bàn phím chưa phản hồi, nhấp vào vùng notebook rồi thử lại. Các nút trên màn hình vẫn dùng được như phương án dự phòng.

In [1]:
import json
import time
from collections import deque
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import math

SEED = 42
GAMMA = 0.95
THETA = 1e-8
MAX_STEPS = 150

EVAL_EPISODES = 1
Q_EPISODES = 12000
SARSA_EPISODES = 12000
MC_EPISODES = 12000
TD_EPISODES = 12000
ALPHA = 0.10
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.9995

REWARDS = {
    "step": -1,
    "key": 5,
    "goal": 20,
    "invalid": -2,
    "recreated_key": 5
}
ACTIONS = [(-1, 0), (1, 0), (0, -1), (0, 1)]
ACTION_NAMES = ["Up", "Down", "Left", "Right"]
ACTION_ARROWS = {"Up": "↑", "Down": "↓", "Left": "←", "Right": "→"}

MAPS = {
    "Map 1 - Basic": [
        "S..W..G",
        ".W.W.W.",
        ".K.DR..",
        ".W.W.W.",
        "...W..."
    ],
    "Map 2 - Intermediate": [
        "S...W..G",
        ".W..W.W.",
        "..K.W...",
        "....DR..",
        ".W..W.W.",
        "....W...",
        ".W......",
        "....W..."
    ],
    "Map 3 - Advanced": [
        "S....W...G",
        ".W...W.W..",
        "..K..W....",
        "...W.W....",
        ".W...W.W..",
        ".....DR...",
        ".W...W.W..",
        "...W.W....",
        ".W...W....",
        ".....W...."
    ]
}

RESULTS_DIR = Path("Key_Door_Grid_World") / "results"
FIGURES_DIR = Path("Key_Door_Grid_World") / "figures"
MAPS_DIR = Path("Key_Door_Grid_World") / "maps"
for folder in (RESULTS_DIR, FIGURES_DIR, MAPS_DIR):
    folder.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)

def validate_map(grid):
    if not grid or not grid[0] or any(len(row) != len(grid[0]) for row in grid):
        raise ValueError("All map rows must have the same non-zero width.")
    cells = "".join(grid)
    for symbol in ("S", "K", "D", "R", "G"):
        if cells.count(symbol) != 1:
            raise ValueError(f"Bản đồ phải chứa chính xác 1 {symbol}")
    allowed = set("S G K D R W .".split())
    if any(char not in allowed for char in cells):
        raise ValueError("Bản đồ chứa ký hiệu không được hỗ trợ")
    return len(grid), len(grid[0])

for name, grid in MAPS.items():
    validate_map(grid)

map_legend = {
    "S": "Start",
    "G": "Goal",
    "K": "Original key",
    "D": "Locked door",
    "R": "One-time key recreation",
    "W": "Wall",
    ".": "Empty cell"
}

## Môi trường và luật chơi

In [2]:
class KeyDoorEnv:
    def __init__(self, grid, rewards=None):
        self.grid = list(grid)
        self.rows, self.cols = validate_map(self.grid)
        self.rewards = dict(REWARDS if rewards is None else rewards)
        self.start = self._find("S")
        self.goal = self._find("G")
        self.key_pos = self._find("K")
        self.door_pos = self._find("D")
        self.recreation_pos = self._find("R")
        self.initial_state = (*self.start, False, False, True)

    def _find(self, symbol):
        for r, row in enumerate(self.grid):
            for c, value in enumerate(row):
                if value == symbol:
                    return r, c
        raise ValueError(f"Thiếu biểu tượng trên bản đồ: {symbol}")

    def is_terminal(self, state):
        return state[:2] == self.goal

    def transition(self, state, action):
        if self.is_terminal(state):
            return state, 0.0, "terminal"
        r, c, has_key, door_open, recreation_available = state
        dr, dc = ACTIONS[int(action)]
        nr, nc = r + dr, c + dc

        if not (0 <= nr < self.rows and 0 <= nc < self.cols):
            return state, self.rewards["invalid"], "invalid"
        tile = self.grid[nr][nc]
        if tile == "W":
            return state, self.rewards["invalid"], "invalid"

        next_has_key = has_key
        next_door_open = door_open
        next_recreation_available = recreation_available
        reward = self.rewards["step"]
        event = "move"

        if tile == "D" and not next_door_open:
            if not next_has_key:
                return state, self.rewards["invalid"], "locked_door"
            next_has_key = False
            next_door_open = True
            event = "door_opened"

        if tile == "K" and not next_has_key and not next_door_open:
            next_has_key = True
            reward += self.rewards["key"]
            event = "original_key_collected"

        if tile == "R" and next_door_open and next_recreation_available and not next_has_key:
            next_has_key = True
            next_recreation_available = False
            reward += self.rewards["recreated_key"]
            event = "key_recreated"

        if tile == "G":
            reward += self.rewards["goal"]
            event = "goal"

        next_state = (nr, nc, next_has_key, next_door_open, next_recreation_available)
        return next_state, reward, event

    def reachable_states(self):
        visited = {self.initial_state}
        queue = deque([self.initial_state])
        while queue:
            state = queue.popleft()
            if self.is_terminal(state):
                continue
            for action in range(len(ACTIONS)):
                next_state, _, _ = self.transition(state, action)
                if next_state not in visited:
                    visited.add(next_state)
                    queue.append(next_state)
        return sorted(visited)

def build_model(env, states):
    state_set = set(states)
    model = {}
    for state in states:
        model[state] = []
        for action in range(len(ACTIONS)):
            next_state, reward, event = env.transition(state, action)
            if next_state not in state_set:
                raise RuntimeError("The model reached a state absent from reachable_states().")
            model[state].append((next_state, reward, event))
    return model

def render_map(grid, ax=None, title=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(max(5, len(grid[0]) * 0.65), max(3, len(grid) * 0.55)))
    ax.imshow(np.zeros((len(grid), len(grid[0]))), cmap="Greys", vmin=0, vmax=3)
    colors = {"S": "#b7e4c7", "G": "#90caf9", "K": "#ffe082", "D": "#ef9a9a", "R": "#ce93d8", "W": "#616161", ".": "#ffffff"}
    for r, row in enumerate(grid):
        for c, tile in enumerate(row):
            ax.add_patch(plt.Rectangle((c - .5, r - .5), 1, 1, facecolor=colors[tile], edgecolor="#999999"))
            ax.text(c, r, tile, ha="center", va="center", fontsize=12, color="black" if tile != "W" else "white", fontweight="bold")
    ax.set_xticks(range(len(grid[0])))
    ax.set_yticks(range(len(grid)))
    ax.set_xticklabels([])
    ax.set_yticklabels([])
    ax.grid(False)
    ax.set_title(title or "Key-Door Grid World")
    ax.set_xlim(-0.5, len(grid[0]) - 0.5)
    ax.set_ylim(len(grid) - 0.5, -0.5)
    return ax

## Bàn chơi tương tác

In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import output
from IPython.display import Javascript

game = {
    "env": None,
    "state": None,
    "steps": 0,
    "total_reward": 0.0,
    "last_event": "Chọn bản đồ và nhấn Bắt đầu.",
    "finished": False
}

map_selector = widgets.Dropdown(
    options=list(MAPS.keys()),
    value=list(MAPS.keys())[0],
    description="Bản đồ:",
    layout=widgets.Layout(width="320px")
)

start_button = widgets.Button(
    description="Bắt đầu / Chơi lại",
    button_style="success",
    icon="refresh",
    layout=widgets.Layout(width="180px", height="42px")
)

up_button = widgets.Button(description="↑ Lên", layout=widgets.Layout(width="100px", height="42px"))
down_button = widgets.Button(description="↓ Xuống", layout=widgets.Layout(width="100px", height="42px"))
left_button = widgets.Button(description="← Trái", layout=widgets.Layout(width="100px", height="42px"))
right_button = widgets.Button(description="Phải →", layout=widgets.Layout(width="100px", height="42px"))

status_output = widgets.Output()
board_output = widgets.Output()

def draw_game():
    with board_output:
        clear_output(wait=True)
        env = game["env"]
        state = game["state"]
        fig, ax = plt.subplots(figsize=(max(6, env.cols * 0.85), max(4, env.rows * 0.75)))
        render_map(env.grid, ax=ax, title=f"Key-Door Grid World | {map_selector.value}")
        r, c, has_key, door_open, recreation_available = state
        ax.scatter([c], [r], s=430, marker="o", c="#1565c0", edgecolors="white", linewidths=2, zorder=10)
        ax.text(c, r, "P", color="white", ha="center", va="center", fontweight="bold", fontsize=11, zorder=11)
        ax.set_title(f"{map_selector.value} | Player position: ({r}, {c})")
        fig.tight_layout()
        display(fig)
        plt.close(fig)

    with status_output:
        clear_output(wait=True)
        state = game["state"]
        print(f"Step: {game['steps']} / {MAX_STEPS}")
        print(f"Position: ({state[0]}, {state[1]})")
        print(f"Key: {'Đang giữ' if state[2] else 'Không có'}")
        print(f"Door: {'Đã mở' if state[3] else 'Đang khóa'}")
        print(f"Recreated key available: {'Có' if state[4] else 'Không'}")
        print(f"Last event: {game['last_event']}")
        print(f"Total reward: {game['total_reward']:.1f}")
        if game["finished"]:
            if game["env"].is_terminal(state):
                print("\n🎉 CHIẾN THẮNG! Bạn đã đến đích.")
            else:
                print("\nĐã hết lượt. Nhấn 'Bắt đầu / Chơi lại' để thử lại.")

def start_game(_=None):
    game["env"] = KeyDoorEnv(MAPS[map_selector.value])
    game["state"] = game["env"].initial_state
    game["steps"] = 0
    game["total_reward"] = 0.0
    game["last_event"] = "Bắt đầu tại vị trí S."
    game["finished"] = False
    draw_game()
    update_controls()

def move_player(action):
    if game["env"] is None or game["finished"]:
        return
    next_state, reward, event = game["env"].transition(game["state"], action)
    game["state"] = next_state
    game["steps"] += 1
    game["total_reward"] += reward
    game["last_event"] = f"{event} | Reward: {reward:g}"
    if game["env"].is_terminal(next_state) or game["steps"] >= MAX_STEPS:
        game["finished"] = True
    draw_game()
    update_controls()

def update_controls():
    active = game["env"] is not None and not game["finished"]
    for button in (up_button, down_button, left_button, right_button):
        button.disabled = not active

start_button.on_click(start_game)
up_button.on_click(lambda _: move_player(0))
down_button.on_click(lambda _: move_player(1))
left_button.on_click(lambda _: move_player(2))
right_button.on_click(lambda _: move_player(3))

display(widgets.VBox([
    map_selector,
    start_button,
    widgets.HBox([widgets.Label("Điều khiển:")]),
    widgets.VBox([
        widgets.HBox([widgets.Label("", layout=widgets.Layout(width="100px")), up_button]),
        widgets.HBox([left_button, down_button, right_button])
    ]),
    status_output,
    board_output
]))

keyboard_help = widgets.HTML(
    "<b>Điều khiển bàn phím:</b> W / ↑ = Lên &nbsp; "
    "S / ↓ = Xuống &nbsp; A / ← = Trái &nbsp; D / → = Phải"
)
display(keyboard_help)

def handle_keypress(key):
    key = str(key).lower()
    key_actions = {
        "arrowup": 0, "w": 0,
        "arrowdown": 1, "s": 1,
        "arrowleft": 2, "a": 2,
        "arrowright": 3, "d": 3
    }
    if key in key_actions:
        move_player(key_actions[key])

output.register_callback("keydoor.handle_keypress", handle_keypress)
display(Javascript("""
(() => {
    if (window.keyDoorKeyboardInstalled) return;
    window.keyDoorKeyboardInstalled = true;
    document.addEventListener("keydown", (event) => {
        const key = event.key.toLowerCase();
        const allowed = ["arrowup", "arrowdown", "arrowleft", "arrowright", "w", "a", "s", "d"];
        if (!allowed.includes(key)) return;
        if (["INPUT", "TEXTAREA", "SELECT"].includes(document.activeElement.tagName)) return;
        event.preventDefault();
        google.colab.kernel.invokeFunction("keydoor.handle_keypress", [key], {});
    });
})();
"""))

start_game()


ModuleNotFoundError: No module named 'ipywidgets'